# ESM Embedding Exploration (with variant labels and functionality)

Recent work on AI-Interpretability suggests that we may be able to start identifying features within the latent space of neural networks that describe the abstractions they use to make predictions from relatively unstructured data. Our hope is that this work can be applied to variant effect prediction with foundation models, such as ESM-2, in order to provide more actionable information about recently identified variants than functional prediction alone.

In order to demonstrate the feature extraction abilities of ESM-2, this notebook provides a brief analysis of ESM-2 language modeling embeddings for hemoglobin variants retrieved from the [HbVar database](https://globin.bx.psu.edu/hbvar/hbvar.html). 

In [1]:
# imports
from transformers import AutoTokenizer, TFEsmModel 
import tensorflow as tf
from Bio import SeqIO
import json
import plotly.express as px
import os
import numpy as np
import pandas as pd
from umap import UMAP

# embedding directory
output_directory = "../data/variant_embeddings/all_vars"

# load model
hug_face_id = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(hug_face_id)
model = TFEsmModel.from_pretrained(hug_face_id)

# sanity check
inputs = tokenizer("Hello, my dog is cute", return_tensors="tf")
outputs = model(**inputs)
last_hidden_states = outputs.last_hidden_state

print(last_hidden_states, outputs)

2024-11-08 14:33:27.891954: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-08 14:33:27.902053: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1731098007.913791   66625 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1731098007.917274   66625 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-08 14:33:27.929698: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

tf.Tensor(
[[[ 0.05059963  0.10085721 -0.10023749 ... -0.12190716  0.40376827
    0.08775394]
  [ 0.26290226  0.02058962 -0.16462716 ... -0.05984128  0.3401519
    0.15950891]
  [ 0.13614741  0.17050998  0.10192016 ... -0.17278999  0.32610363
    0.48546052]
  ...
  [ 0.28022113  0.06663495 -0.09451015 ... -0.11177716  0.1806937
    0.5831055 ]
  [ 0.19588457  0.14026381  0.07719216 ... -0.14424258  0.22779934
    0.39744595]
  [ 0.5040827   0.16508391 -0.00226392 ... -0.24705702  0.12229864
    0.42984688]]], shape=(1, 8, 1280), dtype=float32) TFBaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=<tf.Tensor: shape=(1, 8, 1280), dtype=float32, numpy=
array([[[ 0.05059963,  0.10085721, -0.10023749, ..., -0.12190716,
          0.40376827,  0.08775394],
        [ 0.26290226,  0.02058962, -0.16462716, ..., -0.05984128,
          0.3401519 ,  0.15950891],
        [ 0.13614741,  0.17050998,  0.10192016, ..., -0.17278999,
          0.32610363,  0.48546052],
        ...,
        [ 0

In [2]:
# load heme variant sequences
var_data = pd.read_csv("../data/hbvar_metadata_sequence_merge.csv")
var_data.head()


,electrophoresisComment,functionalStudies,hgvsName,chromatography,dnaDescriptionComment,name,aliases,stability,functionalComment,variantComment,Structure,electrophoresis,thalType,hemoglobin_chain,sequence,variant_name
0,NaN,[],HBA2:c.5T>G (or HBA1),{'Cation exchange HPLC (Bio-Rad Variant)': 'No...,NaN,Hb Antananarivo,[],{},Normal P<sub>50</sub> under standard condition...,Hb X about 20%; no N-terminal methionine reten...,"{'Protein analysis, Determination, Amino acid ...",{'Isoelectricfocusing': {'units': 'mm from pos...,NaN,alpha1,MGLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,Hb Antananarivo
1,NaN,[],HBA2:c.5T>G (or HBA1),{'Cation exchange HPLC (Bio-Rad Variant)': 'No...,NaN,Hb Antananarivo,[],{},Normal P<sub>50</sub> under standard condition...,Hb X about 20%; no N-terminal methionine reten...,"{'Protein analysis, Determination, Amino acid ...",{'Isoelectricfocusing': {'units': 'mm from pos...,NaN,alpha2,MGLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,Hb Antananarivo
2,NaN,"[['Oxygen affinity', None, 'Increased']]",HBA2:c.8T>G (or HBA1),"{'Anion exchange, other': 'Hb X can be separat...",NaN,Hb Chongqing,[],{'Unknown test': 'Unstable'},NaN,NaN,"{'Protein analysis, separation method': 'Finge...",{'Capillary electrophoresis (Sebia)': {'commen...,NaN,alpha2,MVRSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,Hb Chongqing
3,NaN,"[['Oxygen affinity', None, 'Increased']]",HBA2:c.8T>G (or HBA1),"{'Anion exchange, other': 'Hb X can be separat...",NaN,Hb Chongqing,[],{'Unknown test': 'Unstable'},NaN,NaN,"{'Protein analysis, separation method': 'Finge...",{'Capillary electrophoresis (Sebia)': {'commen...,NaN,alpha1,MVRSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,Hb Chongqing
4,Separates as a fast-moving Hb J from Hb A at a...,[],HBA2:c.17C>A (or HBA1),"{'Anion exchange, other': 'Hb X can be separat...",NaN,Hb J-Toronto,[],{},NaN,Hb J-Toronto interferes with HbA1c measurement...,"{'Protein analysis, Determination, Amino acid ...",{'Capillary electrophoresis (Sebia)': {'units'...,NaN,alpha2,MVLSPDDKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,Hb J-Toronto


In [3]:
# drop duplicate sequences
var_data.drop_duplicates(subset="sequence", inplace=True)
var_data

,electrophoresisComment,functionalStudies,hgvsName,chromatography,dnaDescriptionComment,name,aliases,stability,functionalComment,variantComment,Structure,electrophoresis,thalType,hemoglobin_chain,sequence,variant_name
0,NaN,[],HBA2:c.5T>G (or HBA1),{'Cation exchange HPLC (Bio-Rad Variant)': 'No...,NaN,Hb Antananarivo,[],{},Normal P<sub>50</sub> under standard condition...,Hb X about 20%; no N-terminal methionine reten...,"{'Protein analysis, Determination, Amino acid ...",{'Isoelectricfocusing': {'units': 'mm from pos...,NaN,alpha1,MGLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,Hb Antananarivo
2,NaN,"[['Oxygen affinity', None, 'Increased']]",HBA2:c.8T>G (or HBA1),"{'Anion exchange, other': 'Hb X can be separat...",NaN,Hb Chongqing,[],{'Unknown test': 'Unstable'},NaN,NaN,"{'Protein analysis, separation method': 'Finge...",{'Capillary electrophoresis (Sebia)': {'commen...,NaN,alpha2,MVRSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,Hb Chongqing
4,Separates as a fast-moving Hb J from Hb A at a...,[],HBA2:c.17C>A (or HBA1),"{'Anion exchange, other': 'Hb X can be separat...",NaN,Hb J-Toronto,[],{},NaN,Hb J-Toronto interferes with HbA1c measurement...,"{'Protein analysis, Determination, Amino acid ...",{'Capillary electrophoresis (Sebia)': {'units'...,NaN,alpha2,MVLSPDDKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,Hb J-Toronto
6,NaN,[],HBA2:c.16G>C (or HBA1),{},NaN,Hb Karachi,[],{'Unknown test': 'Normal'},NaN,NaN,"{'Protein analysis, Determination, Amino acid ...",{'Cellulose acetate (alkaline pH)': {'units': ...,NaN,alpha2,MVLSPPDKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,Hb Karachi
8,NaN,"[['Oxygen affinity', None, 'Increased'], ['Boh...",HBA2:c.20A>C (or HBA1),{'DEAE-Sephadex': 'Hb X elutes as a shoulder i...,NaN,Hb Sawara,[],{'Unknown test': 'Normal'},NaN,NaN,"{'Protein analysis, separation method': 'Finge...","{'Starch gel (alkaline pH)': {'units': '', 'va...",NaN,alpha2,MVLSPAAKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,Hb Sawara
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1429,NaN,"[['Oxygen affinity', 'Whole blood', 'Slightly ...",HBB:c.121A>G,{'Cation exchange HPLC': 'HbX elutes before Hb...,NaN,Hb Montpellier,[],{},p50 measured at 21 mmHg on Hemox analyzer.,Hb Montpellier seems to be a high oxygen affin...,{},"{'Capillary electrophoresis': {'value': '', 'c...",NaN,beta,MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQGFFESF...,Hb Montpellier
1430,NaN,[],HBD:c.230C>A,{},NaN,Hb A2-Moyen-Orient,[],{},NaN,No abnormality for the hematological and bioch...,{},{},delta (0 or + unclear),delta,MVHLTPEEKTAVNALWGKVNVDAVGGEALGRLLVVYPWTQRFFESF...,Hb A2-Moyen-Orient
1431,NaN,[],HBB:c.316C>G,{},NaN,Hb Odder,[],{},NaN,This variant emerged incidentally during routi...,{},{'Capillary electrophoresis': {'comment': 'HbX...,NaN,beta,MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESF...,Hb Odder
1432,NaN,[],HBD:c.238G>A,{'Cation exchange HPLC (Bio-Rad Variant)': 'Hb...,NaN,Hb A2-Guangxi,[],{},NaN,NaN,{},"{'Capillary electrophoresis': {'units': '', 'v...",NaN,delta,MVHLTPEEKTAVNALWGKVNVDAVGGEALGRLLVVYPWTQRFFESF...,Hb A2-Guangxi


In [4]:
# generate embeddings
if os.path.exists(output_directory):
    pass
else:
    os.mkdir(output_directory)

max_length = max([len(seq) for seq in var_data["sequence"]])
print(f"Maximum Sequence Length (Embedding Size): {max_length}")

print("Generating Embeddings")
count = 0
for peptide in var_data["sequence"]:
    input = tokenizer(peptide, return_tensors="tf", padding=True, truncation=True, max_length=max_length)
    output = model(**input)
    with open(f"{output_directory}/{peptide}.npy", "wb") as file:
        np.save(file, np.array(output.last_hidden_state))
    count += 1
    if (count%100) == 0:
        print(f"{count} sequence embeddings generated")
print("========================================")
print(f"{count} total sequence embeddings")

Maximum Sequence Length (Embedding Size): 148
Generating Embeddings
100 sequence embeddings generated
200 sequence embeddings generated
300 sequence embeddings generated
400 sequence embeddings generated
500 sequence embeddings generated
600 sequence embeddings generated
700 sequence embeddings generated
800 sequence embeddings generated
900 sequence embeddings generated
1000 sequence embeddings generated
1100 sequence embeddings generated
1200 sequence embeddings generated
1206 total sequence embeddings


# Plot UMAPs of Embeddings

In [5]:
# load embeddings
#embeddings = []
#for file in os.listdir(output_directory):
#    with open(output_directory+"/"+file, "rb") as f:
#        embeddings.append(np.load(f))

# explicitly match embeddings to sequences when re-loading
embeddings = [np.load(f"{output_directory}/{j}.npy") for j in var_data["sequence"]]

# reduce jagged tensors and add to dataframe
def fix_jagged_2d(embed_list):
    sir_gallahad_the_chaste = []
    # find max length in 2nd dim (sequence length)
    max_len = max([len(arr[0]) for arr in embed_list])
    # append zero vector to embeddings
    for i, arr in enumerate(embed_list):
        if len(arr[0]) < max_len:
            padded = np.append(arr[0], np.zeros((max_len - len(arr[0]), 1280)), axis=0)
            sir_gallahad_the_chaste.append(np.expand_dims(padded, axis=0))
        else:
            sir_gallahad_the_chaste.append(arr)
    return sir_gallahad_the_chaste

embeddings = np.concatenate(fix_jagged_2d(embeddings))
print(embeddings.shape)

(1206, 148, 1280)


In [6]:
var_data["embeddings"]=[i for i in embeddings]
var_data["embeddings"][0].shape

(148, 1280)

### ESM-2 Vectors
- To plot on UMAP we must reduce the embeddings to shape: (n-variants, feature_vector)
- The simplest way to achieve this is summing across either the residues, or the 1280-d embeddings for those residues
- Resulting in either a dense sequence representation, or a dense feature representation

In [7]:
len(var_data["embeddings"])

1206

In [8]:
umap = UMAP(n_components=2)
example = var_data["embeddings"][0]

sequence_umap = umap.fit_transform(example)
esm_umap = umap.fit_transform(example.T)

sequence_rep = px.scatter(sequence_umap, x=0,y=1, title="Sequence Rep")
sequence_rep.update_traces(marker_size=3)
esm_rep = px.scatter(esm_umap, x=0,y=1, title="ESM2 Rep")
esm_rep.update_traces(marker_size=3)

sequence_rep.show()
esm_rep.show()

A good way to think about this is imagining the features we are plotting as having either a per-residue basis, or a per-sequence basis. In this sense, the UMAP plots across the entire dataset will reflect ESM-2 representations of the 1280 "features" in its output for each residue, OR, a condensed vector representing the embedding of all features across the sequence.

A more literal interpretation is that the model embeds each sequence in 2 dimensions and which one we choose to slice across for UMAP purposes determines what that UMAP is showing you.

In [9]:
var_data["sequence_representation"] = var_data["embeddings"].apply(lambda x: np.add.reduce(x, axis=-1))
var_data["esm_representation"] = var_data["embeddings"].apply(lambda x: np.add.reduce(x, axis=-2))
print(var_data["sequence_representation"][0].shape)
print(var_data["esm_representation"][0].shape)
print(var_data["embeddings"][0].shape)

(148,)
(1280,)
(148, 1280)


In [10]:
# save data
var_data.to_csv("../data/hbvar_with_embeddings.csv", index=False)

### Sequence Representations

In [11]:
seq_rep = pd.DataFrame(umap.fit_transform(np.array(var_data["sequence_representation"]).tolist()))
seq_rep["label"] = var_data["name"]
seq_rep["color"] = var_data["hemoglobin_chain"]

fig = px.scatter(
    seq_rep, 
    x=0, 
    y=1, 
    color="color", 
    hover_data="label", 
    title="Sequence Representation by Chain"
)
fig.update_traces(marker_size=5)
fig.show()

In [12]:
seq_rep = pd.DataFrame(umap.fit_transform(np.array(var_data["sequence_representation"]).tolist()))
seq_rep["label"] = var_data["name"]
seq_rep["color"] = var_data["thalType"]

fig = px.scatter(
    seq_rep, 
    x=0, 
    y=1, 
    color="color", 
    hover_data="label", 
    title="Sequence Representation by Thalassemia Type"
)
fig.update_traces(marker_size=5)
fig.show()

### ESM2 Representation

In [13]:
seq_rep = pd.DataFrame(umap.fit_transform(np.array(var_data["esm_representation"]).tolist()))
seq_rep["label"] = var_data["name"]
seq_rep["color"] = var_data["hemoglobin_chain"]

fig = px.scatter(
    seq_rep, 
    x=0, 
    y=1, 
    color="color", 
    hover_data="label", 
    title="Sequence Representation by Chain"
)
fig.update_traces(marker_size=5)
fig.show()

In [14]:
seq_rep = pd.DataFrame(umap.fit_transform(np.array(var_data["esm_representation"]).tolist()))
seq_rep["label"] = var_data["name"]
seq_rep["color"] = var_data["thalType"]

fig = px.scatter(
    seq_rep, 
    x=0, 
    y=1, 
    color="color", 
    hover_data="label", 
    title="Sequence Representation by Thalassemia Type"
)
fig.update_traces(marker_size=5)
fig.show()

### Functional Analysis

In theory, embeddings from the ESM-2 stem should demonstrate clustering that aligns with biological features (i.e. model abstractions should align with biological knowledge). However, due to the effect of Superposition, while there is tight clustering in our above examples, the clusters do not align with simple features such as the chain of hemoglobin that the sequence encodes on either a per-residue basis or a per-sequence basis.

Below, we'll explore correlations with functional annotations. See function_annotation script for details on unpacking functional annotations from HbVar.

In [17]:
os.system("python ../pipelines/function_annotation.py")

Finding assay types...
{'Assays not specified': [], 'Cooperativity': [], 'Oxygen affinity': [], 'IHP effect': [], '2,3-DPG effect': [], 'Bohr effect': [], 'Sickling': [], 'Autooxidation': [], 'name': []}
Extracting functional annotations...
Processed 100 variants
Processed 200 variants
Processed 300 variants
Processed 400 variants
Processed 500 variants
Processed 600 variants
Processed 700 variants
Processed 800 variants
Processed 900 variants
Processed 1000 variants
Processed 1100 variants
Processed 1200 variants
Processed 1300 variants
Processed 1400 variants
Results:
  Assays not specified Cooperativity  ... Autooxidation             name
0                 None          None  ...          None  Hb Antananarivo
1                 None          None  ...          None     Hb Chongqing
2                 None          None  ...          None     Hb J-Toronto
3                 None          None  ...          None       Hb Karachi
4                 None          None  ...          None   

0

In [18]:
var_data = pd.read_csv("../data/hbvar-w-func-embed-seq.csv")
# reload embeddings from numpy binaries

var_data.drop(columns=["embeddings", "sequence_representation", "esm_representation"], inplace=True)
var_data["embeddings"]=[i for i in embeddings]
var_data["sequence_representation"] = var_data["embeddings"].apply(lambda x: np.add.reduce(x, axis=-1))
var_data["esm_representation"] = var_data["embeddings"].apply(lambda x: np.add.reduce(x, axis=-2))

var_data.head()

,electrophoresisComment,hgvsName,chromatography,dnaDescriptionComment,name,aliases,stability,functionalComment,variantComment,Structure,...,Cooperativity,Oxygen affinity,IHP effect,"2,3-DPG effect",Bohr effect,Sickling,Autooxidation,embeddings,sequence_representation,esm_representation
0,NaN,HBA2:c.5T>G (or HBA1),{'Cation exchange HPLC (Bio-Rad Variant)': 'No...,NaN,Hb Antananarivo,[],{},Normal P<sub>50</sub> under standard condition...,Hb X about 20%; no N-terminal methionine reten...,"{'Protein analysis, Determination, Amino acid ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[[0.039879776537418365, -0.060473520308732986,...","[-1.4033864405937493, -1.7326773980166763, -1....","[-1.4536067070439458, -19.541362036485225, -22..."
1,NaN,HBA2:c.8T>G (or HBA1),"{'Anion exchange, other': 'Hb X can be separat...",NaN,Hb Chongqing,[],{'Unknown test': 'Unstable'},NaN,NaN,"{'Protein analysis, separation method': 'Finge...",...,NaN,Increased,NaN,NaN,NaN,NaN,NaN,"[[0.04409508407115936, -0.03374907746911049, 0...","[-1.4045900725759566, -1.7490782651584595, -1....","[-0.32130681350827217, -17.899344491772354, -2..."
2,Separates as a fast-moving Hb J from Hb A at a...,HBA2:c.17C>A (or HBA1),"{'Anion exchange, other': 'Hb X can be separat...",NaN,Hb J-Toronto,[],{},NaN,Hb J-Toronto interferes with HbA1c measurement...,"{'Protein analysis, Determination, Amino acid ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[[0.04105670005083084, -0.047855351120233536, ...","[-1.4152285165619105, -1.567654078360647, -1.5...","[-0.9730507722124457, -19.20025932090357, -22...."
3,NaN,HBA2:c.16G>C (or HBA1),{},NaN,Hb Karachi,[],{'Unknown test': 'Normal'},NaN,NaN,"{'Protein analysis, Determination, Amino acid ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[[0.0424206405878067, -0.047186315059661865, 0...","[-1.4001028012717143, -1.5338792277034372, -1....","[-0.7020381223410368, -18.290798322763294, -22..."
4,NaN,HBA2:c.20A>C (or HBA1),{'DEAE-Sephadex': 'Hb X elutes as a shoulder i...,NaN,Hb Sawara,[],{'Unknown test': 'Normal'},NaN,NaN,"{'Protein analysis, separation method': 'Finge...",...,NaN,Increased,NaN,NaN,Normal,NaN,NaN,"[[0.04741267114877701, -0.04828247055411339, -...","[-1.4080889185424894, -1.5772056402638555, -1....","[-0.8190737590193748, -18.023886519484222, -21..."


In [19]:
# filter for variants affecting oxygen affinity
o2_affinity = var_data.dropna(subset="Oxygen affinity")
print(o2_affinity["Oxygen affinity"].value_counts())

Oxygen affinity
Increased             188
Normal                 73
Decreased              58
Slightly increased     22
Slightly decreased      9
Greatly increased       6
Greatly decreased       2
Name: count, dtype: int64


In [20]:
# sequence by function
seq_rep = pd.DataFrame(umap.fit_transform(np.array(o2_affinity["sequence_representation"]).tolist()))
seq_rep["label"] = o2_affinity["name"]
seq_rep["color"] = o2_affinity["Oxygen affinity"]

fig = px.scatter(
    seq_rep, 
    x=0, 
    y=1, 
    color="color", 
    hover_data="label", 
    title="Sequence Representation by Oxygen Affinity"
)
fig.update_traces(marker_size=5)
fig.show()

In [21]:
# esm by function
seq_rep = pd.DataFrame(umap.fit_transform(np.array(o2_affinity["esm_representation"]).tolist()))
seq_rep["label"] = o2_affinity["name"]
seq_rep["color"] = o2_affinity["Oxygen affinity"]

fig = px.scatter(
    seq_rep, 
    x=0, 
    y=1, 
    color="color", 
    hover_data="label", 
    title="ESM-2 Representation by Oxygen Affinity"
)
fig.update_traces(marker_size=5)
fig.show()